# LeetCode 157 — Read N Characters Given Read4 bằng Python

**Link đề:** https://leetcode.com/problems/read-n-characters-given-read4/
**Độ khó:** Easy · **Chủ đề:** Array, Simulation, Interactive (gọi API có sẵn)

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
"""
The read4 API is already defined for you.
    @param buf4, a list of characters
    @return an integer
    def read4(buf4):
"""
class Solution:
    def read(self, buf, n):
        """
        :type buf: Destination buffer (List[str])
        :type n: Number of characters to read (int)
        :rtype: The number of actual characters read (int)
        """
```

## Mục tiêu học
1. Hiểu bài toán dạng **"interactive / API"**: không đọc file trực tiếp, chỉ được gọi hàm `read4`.
2. Hiểu **tham số đầu ra** (output parameter): hàm ghi kết quả vào list `buf` được truyền vào, đồng thời `return` số ký tự.
3. Tự mô phỏng `read4` trong notebook (con trỏ file, đọc từng khối 4 ký tự).
4. Viết lời giải đọc theo khối: chép `min(count, n - total)` ký tự, dừng khi **đủ `n`** hoặc **hết file**.
5. Trace từng lần gọi `read4`, chứng minh đúng, phân tích Time/Space, và mở rộng sang bài 158 (gọi `read` nhiều lần).

## 1. Đề bài (viết lại theo hiểu biết)

Có một **file** chỉ đọc được thông qua API `read4(buf4)`:
- `read4` đọc **tối đa 4 ký tự liên tiếp** từ vị trí hiện tại của file, ghi chúng vào list `buf4` (độ dài 4), và **trả về số ký tự thực sự đọc được** (`0..4`).
- File có một **con trỏ** (file pointer) ngầm: mỗi lần gọi `read4`, con trỏ tiến lên số ký tự vừa đọc. Hết file thì `read4` trả `0`.

Hãy cài đặt `read(buf, n)`:
- Đọc **tối đa `n` ký tự** từ file, ghi vào list `buf` (đã được cấp sẵn đủ chỗ, `buf[0]`, `buf[1]`, ...).
- **Trả về** số ký tự thực sự đọc được (ít hơn `n` nếu file ngắn hơn).

Ghi chú của đề: không được truy cập file trực tiếp; trong bài 157, `read` chỉ được gọi **một lần** cho mỗi test.

| File | `n` | Trả về | `buf` (phần đầu) |
|---|:-:|:-:|---|
| `"abc"` | 4 | 3 | `['a','b','c']` |
| `"abcde"` | 5 | 5 | `['a','b','c','d','e']` |
| `"abcdABCD1234"` | 12 | 12 | toàn bộ file |
| `"leetcode"` | 5 | 5 | `['l','e','e','t','c']` |

**Constraints (theo hiểu biết):** `1 <= len(file) <= 500`, file gồm chữ cái tiếng Anh và chữ số; `1 <= n <= 1000`.

## 2. Nền tảng: list là đối tượng **thay đổi được** (mutable), truyền vào hàm là truyền **tham chiếu**

Khi gọi `f(buf)`, hàm nhận **cùng một list** chứ không phải bản sao. Hàm sửa `buf[i] = ...` thì người gọi **thấy được** sự thay đổi. Đây là cách `read4` "trả" dữ liệu qua `buf4`, và cách `read` trả dữ liệu qua `buf`.

Nhưng nếu trong hàm ta **gán lại tên** `buf = [...]`, đó là tạo list mới gắn vào tên cục bộ — người gọi **không** thấy.

In [1]:
def fill_in_place(buf):
    buf[0] = 'X'           # sửa phần tử -> người gọi thấy

def rebind(buf):
    buf = ['Y', 'Y']       # gán lại tên cục bộ -> người gọi KHÔNG thấy

data = ['a', 'b']
fill_in_place(data)
print("Sau fill_in_place:", data)
rebind(data)
print("Sau rebind:", data)

Sau fill_in_place: ['X', 'b']
Sau rebind: ['X', 'b']


### 2.1. Ghi một đoạn bằng slice assignment

`buf[i:j] = other` thay đoạn `buf[i:j]` bằng các phần tử của `other` — vẫn là sửa **tại chỗ** list `buf`. Nếu `len(other) == j - i` thì độ dài `buf` không đổi.

`buf4[:k]` lấy `k` phần tử đầu của `buf4`.

In [2]:
buf = [''] * 8
buf4 = ['a', 'b', 'c', 'd']
k = 3
buf[2:2 + k] = buf4[:k]
print(buf)
print("buf4[:3] =", buf4[:3], "| buf4[:0] =", buf4[:0])

['', '', 'a', 'b', 'c', '', '', '']
buf4[:3] = ['a', 'b', 'c'] | buf4[:0] = []


## 3. Tự mô phỏng API `read4`

Trên LeetCode, `read4` có sẵn. Trong notebook ta tự viết để chạy được:
- Lớp `File4` giữ nội dung và con trỏ `pos`.
- Method `read4(buf4)` chép tối đa 4 ký tự từ `pos` vào `buf4`, tăng `pos`, trả số ký tự.
- Đếm `calls` để quan sát số lần gọi.

Để `Solution.read` gọi được `read4(buf4)` **như trên LeetCode** (hàm toàn cục), ta tạo hàm toàn cục `read4` chuyển tiếp tới file đang dùng (`CURRENT`). Hàm `set_file(text)` đặt file mới cho mỗi test.

In [3]:
class File4:
    def __init__(self, content: str):
        self.content = content
        self.pos = 0          # con trỏ file
        self.calls = 0        # số lần read4 được gọi

    def read4(self, buf4: list) -> int:
        self.calls += 1
        chunk = self.content[self.pos:self.pos + 4]
        for i, ch in enumerate(chunk):
            buf4[i] = ch
        self.pos += len(chunk)
        return len(chunk)

CURRENT = File4("")

def set_file(text: str) -> File4:
    global CURRENT            # 'global' để gán lại biến toàn cục từ trong hàm
    CURRENT = File4(text)
    return CURRENT

def read4(buf4: list) -> int:  # giống chữ ký API trên LeetCode
    return CURRENT.read4(buf4)

f = set_file("abcdefghijk")
buf4 = [''] * 4
for _ in range(4):
    cnt = read4(buf4)
    print(f"read4 -> {cnt}, buf4 = {buf4}, con trỏ ở vị trí {f.pos}")

read4 -> 4, buf4 = ['a', 'b', 'c', 'd'], con trỏ ở vị trí 4
read4 -> 4, buf4 = ['e', 'f', 'g', 'h'], con trỏ ở vị trí 8
read4 -> 3, buf4 = ['i', 'j', 'k', 'h'], con trỏ ở vị trí 11
read4 -> 0, buf4 = ['i', 'j', 'k', 'h'], con trỏ ở vị trí 11


Quan sát quan trọng:
- Lần thứ 3 chỉ đọc được 3 ký tự (`'i','j','k'`); ô `buf4[3]` vẫn còn **rác** `'h'` từ lần trước. Vì vậy chỉ được dùng `buf4[:cnt]`, **không** dùng cả `buf4`.
- Lần thứ 4 trả `0`: hết file.
- `read4` trả về `< 4` là dấu hiệu **đã chạm cuối file**.

### 3.1. `global` — gán biến toàn cục từ trong hàm

Đọc biến toàn cục trong hàm thì không cần gì. Nhưng **gán** (`CURRENT = ...`) trong hàm sẽ tạo biến cục bộ mới, trừ khi khai báo `global CURRENT`. Trong code nộp LeetCode ta không cần `global` vì không gán `read4`.

In [4]:
counter = 0
def bump_without_global():
    try:
        counter += 1           # gán -> Python coi counter là biến cục bộ chưa có giá trị
    except UnboundLocalError as e:
        print("Lỗi:", e)

def bump_with_global():
    global counter
    counter += 1

bump_without_global()
bump_with_global()
print("counter =", counter)

Lỗi: cannot access local variable 'counter' where it is not associated with a value
counter = 1


## 4. Cách 1 — Baseline: đọc hết file rồi cắt `n` ký tự

Ý nghĩ đơn giản: "gọi `read4` cho tới khi trả 0, gom hết vào một list tạm, rồi chép `n` ký tự đầu sang `buf`".

Đúng kết quả, nhưng:
- đọc **thừa** khi `n` nhỏ hơn nhiều so với file (gọi `read4` quá số lần cần);
- tốn bộ nhớ `O(độ dài file)` cho list tạm;
- trong bài 158 (gọi `read` nhiều lần), đọc lố sẽ **làm mất** dữ liệu cho lần sau.

In [5]:
def read_all_then_cut(buf: list, n: int) -> int:
    everything = []
    buf4 = [''] * 4
    while True:
        cnt = read4(buf4)
        if cnt == 0:
            break
        everything.extend(buf4[:cnt])     # extend: nối các phần tử vào cuối list
    k = min(n, len(everything))
    buf[:k] = everything[:k]
    return k

f = set_file("abcdABCD1234")
buf = [''] * 5
print("trả về:", read_all_then_cut(buf, 5), "| buf:", buf, "| số lần gọi read4:", f.calls)

trả về: 5 | buf: ['a', 'b', 'c', 'd', 'A'] | số lần gọi read4: 4


Với `n = 5`, ta chỉ cần 2 lần `read4` (đọc 8 ký tự là đủ để có 5), nhưng baseline gọi **4** lần (3 lần có dữ liệu + 1 lần trả 0). Ta muốn dừng **đúng lúc**.

## 5. Insight — hai điều kiện dừng và "chỉ lấy phần cần"

Duyệt theo **khối 4 ký tự**. Giữ `total` = số ký tự đã ghi vào `buf`.

Mỗi vòng:
1. `cnt = read4(buf4)` — số ký tự khối này có.
2. Chỉ lấy `take = min(cnt, n - total)` — không vượt quá phần còn thiếu.
3. Chép `buf4[:take]` vào `buf[total : total + take]`, `total += take`.
4. **Dừng** khi:
   - `total == n` (đã đủ), **hoặc**
   - `cnt < 4` (hết file — khối này không đầy).

```text
file "abcde", n = 5
vòng 1: cnt = 4 ('abcd'), cần 5 - 0 = 5 -> take 4 -> total 4
vòng 2: cnt = 1 ('e'),    cần 5 - 4 = 1 -> take 1 -> total 5 -> đủ, dừng
```

In [6]:
for cnt, total, n in [(4, 0, 5), (1, 4, 5), (4, 8, 10), (3, 0, 4)]:
    print(f"cnt={cnt}, total={total}, n={n} -> take = min({cnt}, {n - total}) = {min(cnt, n - total)}")

cnt=4, total=0, n=5 -> take = min(4, 5) = 4
cnt=1, total=4, n=5 -> take = min(1, 1) = 1
cnt=4, total=8, n=10 -> take = min(4, 2) = 2
cnt=3, total=0, n=4 -> take = min(3, 4) = 3


## 6. Cách 2 — Lời giải chính: đọc từng khối, dừng sớm

Từng đoạn:

**Đoạn 1 — chuẩn bị**
```python
buf4 = [''] * 4      # bộ đệm tạm 4 ô
total = 0
```

**Đoạn 2 — lặp khi chưa đủ**
```python
while total < n:
    cnt = read4(buf4)
```

**Đoạn 3 — chỉ lấy phần cần**
```python
    take = min(cnt, n - total)
    buf[total:total + take] = buf4[:take]
    total += take
```

**Đoạn 4 — hết file thì dừng**
```python
    if cnt < 4:
        break
return total
```

In [7]:
def read_blocks(buf: list, n: int) -> int:
    buf4 = [''] * 4
    total = 0
    while total < n:
        cnt = read4(buf4)
        take = min(cnt, n - total)
        buf[total:total + take] = buf4[:take]
        total += take
        if cnt < 4:
            break
    return total

for text, n in [("abc", 4), ("abcde", 5), ("abcdABCD1234", 12), ("leetcode", 5)]:
    f = set_file(text)
    buf = [''] * n
    k = read_blocks(buf, n)
    print(f"file={text!r:<16} n={n:>2} -> trả {k:>2}, buf[:k]={''.join(buf[:k])!r:<15} read4 gọi {f.calls} lần")

file='abc'            n= 4 -> trả  3, buf[:k]='abc'           read4 gọi 1 lần
file='abcde'          n= 5 -> trả  5, buf[:k]='abcde'         read4 gọi 2 lần
file='abcdABCD1234'   n=12 -> trả 12, buf[:k]='abcdABCD1234'  read4 gọi 3 lần
file='leetcode'       n= 5 -> trả  5, buf[:k]='leetc'         read4 gọi 2 lần


`''.join(list_ký_tự)` nối các ký tự trong list thành một chuỗi — chỉ dùng để in cho dễ đọc.

## 7. TRACE từng lần gọi `read4`

### 7.1. Bảng trace `file = "abcdABCD1234"`, `n = 10`

| Vòng | `total` trước | `cnt` | `buf4[:cnt]` | `n - total` | `take` | `total` sau | Dừng? |
|---:|:-:|:-:|---|:-:|:-:|:-:|---|
| 1 | 0 | 4 | `abcd` | 10 | 4 | 4 | không |
| 2 | 4 | 4 | `ABCD` | 6 | 4 | 8 | không |
| 3 | 8 | 4 | `1234` | 2 | **2** | 10 | `total == n` |

Vòng 3 cho thấy nhánh "đọc được 4 nhưng chỉ lấy 2". Hai ký tự `'3','4'` bị bỏ — trong bài 157 không sao vì `read` chỉ gọi một lần.

In [8]:
def trace_read(text: str, n: int) -> int:
    f = set_file(text)
    buf = [''] * n
    buf4 = [''] * 4
    total, step = 0, 1
    while total < n:
        cnt = read4(buf4)
        take = min(cnt, n - total)
        buf[total:total + take] = buf4[:take]
        print(f"Vòng {step}: read4 -> {cnt} {buf4[:cnt]}, cần {n - total}, lấy {take}, "
              f"total {total} -> {total + take}, buf = {''.join(buf[:total + take])!r}")
        total += take
        step += 1
        if cnt < 4:
            print("  cnt < 4 -> hết file, dừng")
            break
    if total == n:
        print("  total == n -> đủ, dừng")
    print(f"Trả về {total}; read4 được gọi {f.calls} lần")
    return total

trace_read("abcdABCD1234", 10)

Vòng 1: read4 -> 4 ['a', 'b', 'c', 'd'], cần 10, lấy 4, total 0 -> 4, buf = 'abcd'
Vòng 2: read4 -> 4 ['A', 'B', 'C', 'D'], cần 6, lấy 4, total 4 -> 8, buf = 'abcdABCD'
Vòng 3: read4 -> 4 ['1', '2', '3', '4'], cần 2, lấy 2, total 8 -> 10, buf = 'abcdABCD12'
  total == n -> đủ, dừng
Trả về 10; read4 được gọi 3 lần


10

### 7.2. Trace nhánh "hết file trước khi đủ `n`" — `file = "abc"`, `n = 4`

In [9]:
trace_read("abc", 4)

Vòng 1: read4 -> 3 ['a', 'b', 'c'], cần 4, lấy 3, total 0 -> 3, buf = 'abc'
  cnt < 4 -> hết file, dừng
Trả về 3; read4 được gọi 1 lần


3

### 7.3. Trace trường hợp đặc biệt: độ dài file **chia hết cho 4** và `n` lớn hơn file

`file = "abcd"`, `n = 8`: vòng 1 đọc đủ 4 (chưa biết đã hết file), vòng 2 `read4` trả `0` → `take = 0` → `cnt < 4` → dừng. Đây là lý do điều kiện dừng là `cnt < 4` chứ không phải `cnt == 0` riêng: cả hai trường hợp `cnt = 0..3` đều nghĩa là hết file.

In [10]:
trace_read("abcd", 8)

Vòng 1: read4 -> 4 ['a', 'b', 'c', 'd'], cần 8, lấy 4, total 0 -> 4, buf = 'abcd'
Vòng 2: read4 -> 0 [], cần 4, lấy 0, total 4 -> 4, buf = 'abcd'
  cnt < 4 -> hết file, dừng
Trả về 4; read4 được gọi 2 lần


4

## 8. Cách 3 — Biến thể cách viết: chép từng ký tự

Thay slice assignment bằng vòng `for` chép từng ký tự — dài hơn nhưng giống cách viết trong C/Java và rõ ràng cho người mới. Ta cũng gộp điều kiện dừng vào `while`.

In [11]:
def read_char_by_char(buf: list, n: int) -> int:
    buf4 = [''] * 4
    total = 0
    eof = False
    while total < n and not eof:
        cnt = read4(buf4)
        if cnt < 4:
            eof = True
        i = 0
        while i < cnt and total < n:
            buf[total] = buf4[i]
            total += 1
            i += 1
    return total

for text, n in [("abc", 4), ("abcde", 5), ("abcdABCD1234", 10)]:
    set_file(text)
    buf = [''] * n
    k = read_char_by_char(buf, n)
    print(text, n, "->", k, buf[:k])

abc 4 -> 3 ['a', 'b', 'c']
abcde 5 -> 5 ['a', 'b', 'c', 'd', 'e']
abcdABCD1234 10 -> 10 ['a', 'b', 'c', 'd', 'A', 'B', 'C', 'D', '1', '2']


## 9. Code nộp LeetCode

Trên LeetCode **không** định nghĩa lại `read4` — nó có sẵn. Cell dưới chạy được trong notebook nhờ `read4` toàn cục đã mô phỏng ở mục 3.

In [12]:
class Solution:
    def read(self, buf, n):
        buf4 = [''] * 4
        total = 0
        while total < n:
            cnt = read4(buf4)
            take = min(cnt, n - total)
            buf[total:total + take] = buf4[:take]
            total += take
            if cnt < 4:
                break
        return total

set_file("leetcode")
buf = [''] * 5
print(Solution().read(buf, 5), buf)

5 ['l', 'e', 'e', 't', 'c']


**Lưu ý về `buf`:** LeetCode cấp sẵn `buf` đủ lớn. Slice assignment với `len(buf4[:take]) == take` không làm đổi độ dài `buf`. Nếu muốn tuyệt đối an toàn (một số bản Python của đề dùng `buf` là list có độ dài cố định), có thể chép từng ký tự như cách 3.

## 10. Vì sao đúng? (invariant)

Ở đầu mỗi vòng lặp:

> `buf[0:total]` chứa đúng `total` ký tự **đầu tiên** của file, `total <= n`, và con trỏ file nằm ở vị trí `>= total` (có thể vượt `total` tối đa 3 ký tự ở vòng cuối).

- **Ban đầu:** `total = 0`, chưa đọc gì → đúng.
- **Một vòng:** `read4` trả `cnt` ký tự **tiếp theo** của file (vì các vòng trước luôn lấy hết `cnt` khi chưa đủ — chỉ vòng cuối mới lấy thiếu). Ta chép `take = min(cnt, n - total)` ký tự đầu của khối vào ngay sau `total` → `buf[0:total + take]` vẫn là các ký tự đầu của file, và `total + take <= n`.
- **Dừng:**
  - `total == n`: đã có đủ `n` ký tự đầu → trả `n`, đúng.
  - `cnt < 4`: file đã hết. Khi đó `take = min(cnt, n - total)`; nếu `take == cnt` ta đã lấy hết file → trả độ dài file (`< n`), đúng; nếu `take < cnt` thì `total` vừa đạt `n` → cũng đúng.
- **Kết thúc chắc chắn:** mỗi vòng hoặc `total` tăng, hoặc `cnt < 4` (dừng). Khi `cnt == 4` thì `take >= 1` vì `n - total >= 1`.

Kết quả trả về = `min(n, len(file))`.

## 11. Time / Space — so sánh

Gọi `L` = độ dài file, `m = min(n, L)`.

| Cách | Số lần gọi `read4` | Time | Space phụ | Ghi chú |
|---|---|---|---|---|
| 1. Đọc hết rồi cắt | `⌊L/4⌋ + 1` | `O(L)` | `O(L)` | đọc thừa, hỏng bài 158 |
| 2. Đọc khối, dừng sớm | `⌈m/4⌉` (thêm 1 nếu `L` chia hết cho 4 và `n > L`) | `O(m)` | `O(1)` (buf4 cố định 4 ô) | **chuẩn** |
| 3. Chép từng ký tự | như cách 2 | `O(m)` | `O(1)` | giống C/Java |

`buf` là đầu ra do đề cấp, không tính vào bộ nhớ phụ.

In [13]:
import math
for text, n in [("abcdABCD1234", 5), ("abcdABCD1234", 12), ("abcd", 8), ("a" * 500, 1000), ("a" * 500, 7)]:
    f = set_file(text); buf = [''] * n
    k = Solution().read(buf, n)
    m = min(n, len(text))
    predicted = math.ceil(m / 4) + (1 if len(text) % 4 == 0 and n > len(text) else 0)
    print(f"L={len(text):>3}, n={n:>4}: trả {k:>3}, read4 gọi {f.calls:>3} lần, dự đoán {predicted}")
    assert f.calls == predicted

L= 12, n=   5: trả   5, read4 gọi   2 lần, dự đoán 2
L= 12, n=  12: trả  12, read4 gọi   3 lần, dự đoán 3
L=  4, n=   8: trả   4, read4 gọi   2 lần, dự đoán 2
L=500, n=1000: trả 500, read4 gọi 126 lần, dự đoán 126
L=500, n=   7: trả   7, read4 gọi   2 lần, dự đoán 2


`math.ceil(x)` làm tròn **lên** (ví dụ `ceil(5/4) = 2`).

## 12. Bộ test

Kiểm tra cả **giá trị trả về** và **nội dung `buf[:k]`**, trên các nhóm: ví dụ, `n < L`, `n == L`, `n > L`, `L` chia hết / không chia hết cho 4, `n = 1`, file 1 ký tự, file dài 500.

In [14]:
def run_read(read_fn, text, n):
    f = set_file(text)
    buf = [''] * max(n, 1)
    k = read_fn(buf, n)
    return k, ''.join(buf[:k])

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol_read = Solution().read
check("ví dụ abc, n=4", run_read(sol_read, "abc", 4), (3, "abc"))
check("ví dụ abcde, n=5", run_read(sol_read, "abcde", 5), (5, "abcde"))
check("ví dụ 12 ký tự, n=12", run_read(sol_read, "abcdABCD1234", 12), (12, "abcdABCD1234"))
check("n < L, giữa khối", run_read(sol_read, "leetcode", 5), (5, "leetc"))
check("n = 1", run_read(sol_read, "xyz", 1), (1, "x"))
check("file 1 ký tự, n lớn", run_read(sol_read, "q", 1000), (1, "q"))
check("L chia hết 4, n > L", run_read(sol_read, "abcd", 8), (4, "abcd"))
check("n đúng bằng 4", run_read(sol_read, "abcdefg", 4), (4, "abcd"))
check("file 500 ký tự", run_read(sol_read, "ab" * 250, 1000)[0], 500)

PASS ví dụ abc, n=4: (3, 'abc')
PASS ví dụ abcde, n=5: (5, 'abcde')
PASS ví dụ 12 ký tự, n=12: (12, 'abcdABCD1234')
PASS n < L, giữa khối: (5, 'leetc')
PASS n = 1: (1, 'x')
PASS file 1 ký tự, n lớn: (1, 'q')
PASS L chia hết 4, n > L: (4, 'abcd')
PASS n đúng bằng 4: (4, 'abcd')
PASS file 500 ký tự: 500


In [15]:
import random, string
rng = random.Random(157)
alphabet = string.ascii_letters + string.digits
fns = [read_all_then_cut, read_blocks, read_char_by_char, Solution().read]
for _ in range(3000):
    L = rng.randint(1, 40)
    text = ''.join(rng.choice(alphabet) for _ in range(L))
    n = rng.randint(1, 50)
    expected = (min(n, L), text[:n])
    for fn in fns:
        assert run_read(fn, text, n) == expected, (fn.__name__, text, n)
print("4 cài đặt khớp text[:n] trên 3000 file ngẫu nhiên")
print("string.ascii_letters + digits =", alphabet[:10], "...", len(alphabet), "ký tự")

4 cài đặt khớp text[:n] trên 3000 file ngẫu nhiên
string.ascii_letters + digits = abcdefghij ... 62 ký tự


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chép cả 4 ô của `buf4` (lấy cả rác)

In [16]:
def read_bug_copy_all4(buf, n):
    buf4 = [''] * 4
    total = 0
    while total < n:
        cnt = read4(buf4)
        buf[total:total + 4] = buf4          # SAI: chép cả 4 ô, kể cả rác
        total += cnt
        if cnt < 4:
            break
    return min(total, n)

f = set_file("abcdefg")
buf = [''] * 8
k = read_bug_copy_all4(buf, 8)
print("trả", k, "| buf =", buf, "-> buf[7] =", repr(buf[7]), "là rác (file chỉ có 7 ký tự)")

trả 7 | buf = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'd'] -> buf[7] = 'd' là rác (file chỉ có 7 ký tự)


Ô `buf[7]` chứa `'d'` — rác còn sót ở `buf4[3]` từ lần đọc thứ nhất (lần 2 chỉ ghi đè 3 ô `'e','f','g'`). Ngoài ra nếu `total + 4` vượt độ dài `buf`, slice assignment còn làm `buf` **dài ra**. Nếu người gọi đọc cả `buf`, họ nhận dữ liệu sai. Sửa: chỉ chép `buf4[:take]`.

### Lỗi 2 — Không giới hạn bởi `n` (trả về/ghi quá `n`)

In [17]:
def read_bug_ignore_n(buf, n):
    buf4 = [''] * 4
    total = 0
    while total < n:
        cnt = read4(buf4)
        buf[total:total + cnt] = buf4[:cnt]   # SAI: lấy hết cnt dù chỉ cần ít hơn
        total += cnt
        if cnt < 4:
            break
    return total

print("Bản lỗi, 'leetcode', n=5 ->", run_read(read_bug_ignore_n, "leetcode", 5), "(đúng phải là (5, 'leetc'))")

Bản lỗi, 'leetcode', n=5 -> (8, 'leetcode') (đúng phải là (5, 'leetc'))


### Lỗi 3 — Chỉ dừng khi `cnt == 0`

```python
while True:
    cnt = read4(buf4)
    if cnt == 0: break
```
Vẫn đúng kết quả nhưng gọi `read4` **thêm một lần thừa** mỗi khi khối cuối không đầy. Với bài 158 hoặc API thật (đọc mạng, đĩa), một lời gọi thừa là chi phí thật. `cnt < 4` đã đủ báo hết file.

### Lỗi 4 — Gán lại `buf = ...` trong hàm
```python
buf = buf4[:take]       # SAI: tạo list mới, người gọi không thấy gì
```
Phải **sửa tại chỗ** (`buf[i] = ...` hoặc `buf[a:b] = ...`).

In [18]:
def read_bug_rebind(buf, n):
    buf4 = [''] * 4
    cnt = read4(buf4)
    buf = buf4[:min(cnt, n)]        # gán lại tên cục bộ
    return len(buf)

print("Bản gán lại buf:", run_read(read_bug_rebind, "abc", 3), "(trả 3 nhưng buf người gọi rỗng)")

Bản gán lại buf: (3, '') (trả 3 nhưng buf người gọi rỗng)


### Lỗi 5 — Tự đọc biến file / nội dung thay vì gọi `read4`
Đề cấm truy cập file trực tiếp. Trên LeetCode không có biến nào chứa nội dung file để đọc — chỉ có `read4`.

## 14. Mẹo nhận dạng pattern

- Đề cho một API đọc **theo khối cố định** (read4, đọc theo trang, theo gói mạng) và yêu cầu đọc **đúng `n`** → vòng lặp khối + `take = min(cnt, còn_thiếu)` + dừng khi **đủ** hoặc **khối không đầy**.
- "Output parameter": kết quả ghi vào list truyền vào → **sửa tại chỗ**, đừng gán lại tên.
- Nếu hàm được gọi **nhiều lần** (bài 158) → phải **giữ lại phần đọc thừa** giữa các lần gọi (bộ đệm là thuộc tính của đối tượng).
- Bài liên quan: 158 Read N Characters Given read4 II (Hard), 341 Flatten Nested List Iterator (giữ trạng thái giữa các lời gọi), 604 Design Compressed String Iterator.

## 15. Biến thể — bài 158: `read` được gọi **nhiều lần**

Khi `read` gọi nhiều lần trên cùng một file, ký tự đọc thừa ở lần trước (ví dụ `'3','4'` ở mục 7.1) phải được trả cho lần sau. Ý tưởng:
- Lưu `self.buf4`, `self.i4` (vị trí đã dùng trong `buf4`), `self.n4` (số ký tự hợp lệ trong `buf4`) làm **thuộc tính đối tượng**.
- Mỗi lần cần ký tự: nếu `self.i4 == self.n4` (bộ đệm đã dùng hết) thì gọi `read4` nạp lại; nếu `read4` trả 0 thì hết file.

In [19]:
class SolutionMultiCall:
    def __init__(self):
        self.buf4 = [''] * 4
        self.i4 = 0      # vị trí tiếp theo cần lấy trong buf4
        self.n4 = 0      # số ký tự hợp lệ trong buf4

    def read(self, buf, n):
        total = 0
        while total < n:
            if self.i4 == self.n4:            # bộ đệm rỗng -> nạp khối mới
                self.n4 = read4(self.buf4)
                self.i4 = 0
                if self.n4 == 0:
                    break                     # hết file
            buf[total] = self.buf4[self.i4]
            total += 1
            self.i4 += 1
        return total

set_file("abc")
s = SolutionMultiCall()
for n in [1, 2, 1]:
    buf = [''] * n
    k = s.read(buf, n)
    print(f"read(n={n}) -> {k}, {buf[:k]}")

read(n=1) -> 1, ['a']
read(n=2) -> 2, ['b', 'c']
read(n=1) -> 0, []


Với bài 157 (gọi một lần) bản này cũng đúng. Ngược lại, bản 157 **không** dùng được cho 158 — thử gọi hai lần sẽ mất ký tự:

In [20]:
set_file("abcdefg")
s157 = Solution()
b1 = [''] * 2; k1 = s157.read(b1, 2)
b2 = [''] * 2; k2 = s157.read(b2, 2)
print("Bản 157 gọi 2 lần:", b1[:k1], b2[:k2], "(mất 'c','d' — đúng phải là ['a','b'] ['c','d'])")

set_file("abcdefg")
s158 = SolutionMultiCall()
b1 = [''] * 2; k1 = s158.read(b1, 2)
b2 = [''] * 2; k2 = s158.read(b2, 2)
print("Bản 158 gọi 2 lần:", b1[:k1], b2[:k2])

Bản 157 gọi 2 lần: ['a', 'b'] ['e', 'f'] (mất 'c','d' — đúng phải là ['a','b'] ['c','d'])
Bản 158 gọi 2 lần: ['a', 'b'] ['c', 'd']


## 16. Bài tập tự luyện

### Bài 1 — Đếm tay
File dài `L = 10`, `n = 7`. Cách 2 gọi `read4` mấy lần? Lần cuối đọc được bao nhiêu và lấy bao nhiêu?

### Bài 2 — Dự đoán
File `"abcdefgh"` (8 ký tự), `n = 100`. Cách 2 gọi `read4` mấy lần và trả về gì?

### Bài 3 — TODO: tự viết `read` (cách 2)

In [21]:
def exercise_read(buf: list, n: int) -> int:
    # TODO 1: buf4 = [''] * 4, total = 0
    # TODO 2: while total < n: cnt = read4(buf4); take = min(cnt, n - total)
    # TODO 3: chép buf4[:take] vào buf, cập nhật total, dừng nếu cnt < 4
    pass

# assert run_read(exercise_read, "abcde", 5) == (5, "abcde")
# assert run_read(exercise_read, "abc", 4) == (3, "abc")

### Bài 4 — TODO: viết `read4` của riêng bạn cho một "file" là list ký tự
`make_reader(chars)` trả về **một hàm** `my_read4(buf4)` có con trỏ riêng (dùng closure + `nonlocal`).

In [22]:
def exercise_make_reader(chars: list):
    pos = 0
    def my_read4(buf4):
        # TODO: nonlocal pos; chép tối đa 4 ký tự từ chars[pos:], tăng pos, trả số ký tự
        pass
    return my_read4

# r = exercise_make_reader(list("hello"))
# b4 = [''] * 4
# assert r(b4) == 4 and b4 == ['h', 'e', 'l', 'l']
# assert r(b4) == 1 and b4[0] == 'o'
# assert r(b4) == 0

### Bài 5 — Biến thể: `readK` với API `readK(bufk)` đọc tối đa `k` ký tự
Tổng quát hóa: viết `read_with_block(buf, n, block_read, k)` dùng `block_read(bufk)` đọc tối đa `k` ký tự.

In [23]:
def exercise_read_with_block(buf: list, n: int, block_read, k: int) -> int:
    # TODO: giống cách 2, thay 4 bằng k
    pass

# (test ở phần đáp án dùng một block_read k = 3)

### Bài 6 — Mini challenge: tự viết bài 158 (gọi nhiều lần)
Không xem mục 15, tự cài đặt class giữ bộ đệm giữa các lần gọi.

In [24]:
class ExerciseMultiCall:
    def __init__(self):
        # TODO: bộ đệm buf4, vị trí i4, số ký tự hợp lệ n4
        pass

    def read(self, buf, n):
        # TODO
        pass

# set_file("abc"); s = ExerciseMultiCall()
# b1 = [''] * 1; assert s.read(b1, 1) == 1 and b1 == ['a']
# b2 = [''] * 2; assert s.read(b2, 2) == 2 and b2 == ['b', 'c']
# b3 = [''] * 1; assert s.read(b3, 1) == 0

## Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Mỗi lần tối đa 4 | 4 + 4 ≥ 7 | lần 2 đọc 4, lấy 3 |
| 2 | Khối cuối đầy → chưa biết hết file | lần 3 trả 0 | 3 lần, trả 8 |
| 3 | Hai điều kiện dừng | `take = min(cnt, n - total)` | `buf[total:total+take] = buf4[:take]` |
| 4 | Hàm con nhớ biến của hàm cha | `nonlocal pos` | `chunk = chars[pos:pos+4]` rồi chép, `pos += len(chunk)` |
| 5 | Thay hằng 4 bằng `k` | `bufk = [''] * k` | dừng khi `cnt < k` |
| 6 | Bộ đệm sống giữa các lần gọi | thuộc tính `self.i4`, `self.n4` | nạp lại khi `i4 == n4`, `n4 == 0` thì hết file |

## Đáp án tham khảo

**Bài 1:** 2 lần. Lần 1 đọc 4 lấy 4 (`total = 4`); lần 2 đọc 4, cần 3 → lấy 3 (`total = 7 = n`) → dừng.

**Bài 2:** 3 lần: đọc 4, đọc 4 (vẫn `cnt == 4` nên chưa biết đã hết file), lần 3 trả 0 → dừng. Trả về 8.

In [25]:
def answer_read(buf, n):
    buf4 = [''] * 4
    total = 0
    while total < n:
        cnt = read4(buf4)
        take = min(cnt, n - total)
        buf[total:total + take] = buf4[:take]
        total += take
        if cnt < 4:
            break
    return total

def answer_make_reader(chars):
    pos = 0
    def my_read4(buf4):
        nonlocal pos                      # gán biến của hàm bao ngoài
        chunk = chars[pos:pos + 4]
        for i, ch in enumerate(chunk):
            buf4[i] = ch
        pos += len(chunk)
        return len(chunk)
    return my_read4

def answer_read_with_block(buf, n, block_read, k):
    bufk = [''] * k
    total = 0
    while total < n:
        cnt = block_read(bufk)
        take = min(cnt, n - total)
        buf[total:total + take] = bufk[:take]
        total += take
        if cnt < k:
            break
    return total

class AnswerMultiCall:
    def __init__(self):
        self.buf4, self.i4, self.n4 = [''] * 4, 0, 0
    def read(self, buf, n):
        total = 0
        while total < n:
            if self.i4 == self.n4:
                self.n4, self.i4 = read4(self.buf4), 0
                if self.n4 == 0:
                    break
            buf[total] = self.buf4[self.i4]
            total += 1
            self.i4 += 1
        return total

# Bài 1, 2
f = set_file("0123456789"); bb = [''] * 7
assert answer_read(bb, 7) == 7 and f.calls == 2
f = set_file("abcdefgh"); bb = [''] * 100
assert answer_read(bb, 100) == 8 and f.calls == 3
# Bài 3
assert run_read(answer_read, "abcde", 5) == (5, "abcde") and run_read(answer_read, "abc", 4) == (3, "abc")
# Bài 4
r = answer_make_reader(list("hello")); b4 = [''] * 4
assert r(b4) == 4 and b4 == ['h', 'e', 'l', 'l']
assert r(b4) == 1 and b4[0] == 'o' and r(b4) == 0
# Bài 5: block_read đọc 3 ký tự mỗi lần
def make_block_reader(text, k):
    pos = 0
    def block_read(bufk):
        nonlocal pos
        chunk = text[pos:pos + k]
        for i, ch in enumerate(chunk):
            bufk[i] = ch
        pos += len(chunk)
        return len(chunk)
    return block_read
for text, n in [("abcdefg", 5), ("ab", 9), ("abcdef", 6)]:
    bb = [''] * n
    k = answer_read_with_block(bb, n, make_block_reader(text, 3), 3)
    assert (k, ''.join(bb[:k])) == (min(n, len(text)), text[:n])
# Bài 6
set_file("abc"); s = AnswerMultiCall()
b1 = [''] * 1; assert s.read(b1, 1) == 1 and b1 == ['a']
b2 = [''] * 2; assert s.read(b2, 2) == 2 and b2 == ['b', 'c']
b3 = [''] * 1; assert s.read(b3, 1) == 0
print("Đáp án tham khảo PASS toàn bộ!")

Đáp án tham khảo PASS toàn bộ!


## Template nhớ nhanh + checklist submit

```python
def read(self, buf, n):
    buf4 = [''] * 4
    total = 0
    while total < n:                         # dừng 1: đủ n
        cnt = read4(buf4)
        take = min(cnt, n - total)           # chỉ lấy phần còn thiếu
        buf[total:total + take] = buf4[:take]
        total += take
        if cnt < 4:                          # dừng 2: hết file
            break
    return total
```

**Checklist:**
- [ ] Chỉ chép `buf4[:take]`, không chép cả 4 ô.
- [ ] `take = min(cnt, n - total)` — không vượt `n`.
- [ ] Dừng khi `total == n` **hoặc** `cnt < 4`.
- [ ] Sửa `buf` tại chỗ, không gán lại tên `buf`.
- [ ] Không định nghĩa lại `read4` khi nộp; giữ chữ ký `read(self, buf, n)`.
- [ ] Đã thử `n < L`, `n > L`, `L` chia hết cho 4.